# 05 - Analisis Final e Integracion

**Objetivo:** Integrar resultados, seleccionar mejores modelos y conclusiones.

---

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, r2_score, mean_absolute_error,
                             mean_squared_error, roc_curve, auc)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
warnings.filterwarnings('ignore')
np.random.seed(42)
os.makedirs('../results/metrics', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)
print('OK Librerias importadas')

OK Librerias importadas


In [2]:
df = pd.read_csv('../data/04_feature/dataset_ml_preparado.csv', encoding='latin-1')
EXCLUDE = ['id_empleado', 'nombre', 'rut', 'departamento', 'cargo',
            'fecha_ingreso', 'salario', 'tipo_contrato', 'jornada',
            'alto_desempeno', 'avg_desempeno', 'score_global']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in EXCLUDE]
X = df[feature_cols].copy()
y_clf = df['alto_desempeno'].astype(int)
y_reg = df['avg_desempeno'].copy()
X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
_, _, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')
print(f'Features ({len(feature_cols)}): {feature_cols}')

Train: 204  |  Test: 51
Features (12): ['avg_tecnicas', 'avg_blandas', 'total_dias_ausencia', 'total_horas_capacitacion', 'antiguedad_anos', 'total_dias_ausencia_norm', 'total_horas_capacitacion_norm', 'antiguedad_anos_norm', 'departamento_encoded', 'cargo_encoded', 'tipo_contrato_encoded', 'jornada_encoded']


Entrenamiento

In [3]:
clf_models = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'DecisionTree':       DecisionTreeClassifier(random_state=42),
    'RandomForest':       RandomForestClassifier(random_state=42, n_estimators=100),
    'GradientBoosting':   GradientBoostingClassifier(random_state=42),
    'KNN':                KNeighborsClassifier(),
    'SVM':                SVC(random_state=42, probability=True),
}
reg_models = {
    'Ridge':            Ridge(random_state=42),
    'Lasso':            Lasso(random_state=42),
    'DecisionTree':     DecisionTreeRegressor(random_state=42),
    'RandomForest':     RandomForestRegressor(random_state=42, n_estimators=100),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
}
clf_pipes, clf_results = {}, []
for name, model in clf_models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    pipe.fit(X_train, y_clf_train)
    clf_pipes[name] = pipe
    y_pred = pipe.predict(X_test)
    cv = cross_val_score(pipe, X, y_clf, cv=5, scoring='f1', n_jobs=-1)
    clf_results.append({
        'Modelo': name,
        'CV_F1': round(cv.mean(), 4),
        'Test_F1': round(f1_score(y_clf_test, y_pred), 4),
        'Test_Acc': round(accuracy_score(y_clf_test, y_pred), 4),
        'Test_Prec': round(precision_score(y_clf_test, y_pred), 4),
        'Test_Recall': round(recall_score(y_clf_test, y_pred), 4),
    })
reg_pipes, reg_results = {}, []
for name, model in reg_models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('reg', model)])
    pipe.fit(X_train, y_reg_train)
    reg_pipes[name] = pipe
    y_pred = pipe.predict(X_test)
    cv = cross_val_score(pipe, X, y_reg, cv=5, scoring='r2', n_jobs=-1)
    reg_results.append({
        'Modelo': name,
        'CV_R2': round(cv.mean(), 4),
        'Test_R2': round(r2_score(y_reg_test, y_pred), 4),
        'Test_MAE': round(mean_absolute_error(y_reg_test, y_pred), 4),
        'Test_RMSE': round(np.sqrt(mean_squared_error(y_reg_test, y_pred)), 4),
    })
df_clf = pd.DataFrame(clf_results).sort_values('Test_F1', ascending=False)
df_reg = pd.DataFrame(reg_results).sort_values('Test_R2', ascending=False)
print('OK Todos los modelos entrenados')

OK Todos los modelos entrenados


Tabla de resumen

In [4]:
print('='*55)
print('CLASIFICACION - Ordenado por Test F1')
print('='*55)
print(df_clf.to_string(index=False))
print()
print('='*55)
print('REGRESION - Ordenado por Test R2')
print('='*55)
print(df_reg.to_string(index=False))
df_clf.to_csv('../results/metrics/05_resumen_clasificacion.csv', index=False)
df_reg.to_csv('../results/metrics/05_resumen_regresion.csv', index=False)
print('OK Tablas guardadas')

CLASIFICACION - Ordenado por Test F1
            Modelo  CV_F1  Test_F1  Test_Acc  Test_Prec  Test_Recall
      RandomForest 0.7099   0.8077    0.8039     0.7778         0.84
               SVM 0.7208   0.7692    0.7647     0.7407         0.80
LogisticRegression 0.7638   0.7547    0.7451     0.7143         0.80
  GradientBoosting 0.6865   0.7241    0.6863     0.6364         0.84
      DecisionTree 0.7244   0.6923    0.6863     0.6667         0.72
               KNN 0.6719   0.6667    0.6471     0.6207         0.72

REGRESION - Ordenado por Test R2
          Modelo   CV_R2  Test_R2  Test_MAE  Test_RMSE
           Lasso -0.0506  -0.0075    1.1859     1.5888
    RandomForest  0.1954  -0.1452    1.2992     1.6939
           Ridge  0.0630  -0.1547    1.2874     1.7009
GradientBoosting  0.0601  -0.2190    1.3543     1.7476
    DecisionTree -0.5638  -0.9130    1.6944     2.1893
OK Tablas guardadas
